# Ensemble forecast: what you get

This chapter shows the raw output of a single FCN3 ensemble run: 8
members, initialized from the same GFS analysis, perturbed only by the
model's own stochasticity (`Zero` perturbation - no explicit IC noise).
Three views of the same forecast:

1. A Robinson-projection animation of one member's 2m temperature field
   over the rollout.
2. Every member's Munich 2m-temperature path individually (a "spaghetti"
   plot) - what the ensemble spread actually looks like member-by-member,
   not just as a summary statistic.
3. The same data as a boxplot meteogram - the summary-statistic view.

Validation of these checks (are the members physically plausible?) is a
separate chapter - see `06_validation_report`.

In [1]:
import shutil

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from e2s.paths import ProjPaths
from e2s.validation import drop_time, lead_time_hours, nearest_point

paths = ProjPaths()
paths.ensure_directories()

MUNICH_LAT, MUNICH_LON = 48.1372, 11.5755
COLOR_MEMBER = "#6E7B8B"
COLOR_MEAN = "#1F5C99"

ds = xr.open_zarr(paths.ensemble_zarr_path)
x_hours = lead_time_hours(ds)

/tmp/ipykernel_68866/3601682260.py:20: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds = xr.open_zarr(paths.ensemble_zarr_path)


## Every member's path, individually

Each line is one ensemble member's 2m temperature at Munich over the
rollout. Unlike a boxplot, this shows whether spread comes from a few
outlier members or a broad, even spread across all of them.

In [2]:
munich = nearest_point(ds, MUNICH_LAT, MUNICH_LON)
t2m_munich = drop_time(munich["t2m"]).transpose("lead_time", "ensemble").compute().values - 273.15

fig, ax = plt.subplots(figsize=(14, 5))
for i in range(t2m_munich.shape[1]):
    ax.plot(x_hours, t2m_munich[:, i], color=COLOR_MEMBER, alpha=0.6, linewidth=1.0)
ax.plot(x_hours, t2m_munich.mean(axis=1), color=COLOR_MEAN, linewidth=2.5, label="Ensemble mean")
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("Temperature (deg C)")
ax.set_title("Munich - 2m temperature, every ensemble member")
ax.legend(loc="best")
ax.grid(True, color="#DDDDDD", linewidth=0.6)
fig.tight_layout()
fig.savefig(paths.ensemble_book_path / "spaghetti_t2m_munich.png", dpi=150, bbox_inches="tight")
plt.show()

```{figure} ../../output/ensemble/book/spaghetti_t2m_munich.png
:name: fig-ensemble-spaghetti-t2m
Munich 2m temperature, one line per ensemble member.
```

## The same data, summarized

A boxplot per lead-time step compresses the eight lines above into a
distribution - easier to scan for a long rollout, at the cost of hiding
which specific member is where. Generated by `04_analyse.py`.

```{figure} ../../output/ensemble/analysis/meteogram_t2m_munich.png
:name: fig-ensemble-meteogram-t2m
Munich 2m temperature ensemble meteogram (boxplot per lead-time step).
```

## One member, animated

A single member's 2m temperature field over the full rollout, Robinson
projection. All 8 members' animations (t2m and 10m wind) are rendered by
`04_analyse.py` into `data/ensemble/gifs/` - regenerable, not tracked in
git. This one is copied into `output/` as the book's representative
example; see `e2s/paths.py`'s `ensemble_gifs_path` docstring for why the
rest stay out of git.

In [3]:
hero_gif_src = paths.ensemble_gifs_path / "member_00" / "t2m_robinson.gif"
hero_gif_dst = paths.ensemble_book_path / "t2m_robinson_member00.gif"
if hero_gif_src.exists():
    shutil.copyfile(hero_gif_src, hero_gif_dst)
    print(f"Copied {hero_gif_src} -> {hero_gif_dst}")
else:
    print(f"[WARN] {hero_gif_src} not found - run 04_analyse.py first.")

Copied /root/e2s-examples/data/ensemble/gifs/member_00/t2m_robinson.gif -> /root/e2s-examples/output/ensemble/book/t2m_robinson_member00.gif


```{figure} ../../output/ensemble/book/t2m_robinson_member00.gif
:name: fig-ensemble-t2m-gif
Member 0's 2m temperature field over the forecast rollout.
```